# 1. 导入相关的库文件

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset

from torchsummary import summary
import math

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    " 词表相关 "
    vocab_size: int = 50257       # tiktoken 使用的是 GPT-2 的词表，大约有 50257 个token
    hidden_size: int = 768        # 词表中每个token的向量表示长度
    max_squence_length: int = 512 # 模型能够处理的最大输入长度

    " 模型结构相关 "
    block_size: int = 6           # block的个数（灰色大框框）
    head_num_per_block: int = 12  # 每个block里面，多头注意力网络中的头数
    head_size: int = 64           # 每个头的维度
                                  # 需要保障 head_num_per_block * head_size = hidden_size 
                                  # 即：每个block内注意力总头数 = 嵌入向量的长度

    " 训练相关 "
    batch_size: int = 32          # 每次训练的样本数量
    dropout:    float = 0.1

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        # 可以训练的Wq、Wk、Wv
        self.key   = nn.Linear(config.hidden_size, config.head_size, bias=False)
        self.query = nn.Linear(config.hidden_size, config.head_size, bias=False)
        self.value = nn.Linear(config.hidden_size, config.head_size, bias=False)
        # 这是一个因果Attention，需要加入mask
        # mask的作用是：在计算注意力时，屏蔽掉后面的token
        # 这样可以保证模型在训练时，只能看到当前token和之前的token
        # torch.tril 创建的是下三角矩阵，其中下三角部分（包括对角线）为1，上三角部分为0
        self.register_buffer("attention_mask", torch.tril(torch.ones(config.max_squence_length, config.max_squence_length)))
        # dropout 网络
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor)-> torch.Tensor:
        # x: (batch_size, seq_size, hidden_size)
        batch_size, seq_size, hidden_size = x.size()
        # 1. 计算Q、K、V
        # Q、K、V的shape都是 (batch_size, seq_size, head_size)
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)  
        # 2. 计算注意力分数
        weight = q @ k.transpose(1, 2) / math.sqrt(k.size(-1))  
        # 3. 加入mask
        # 通过将上三角部分（代表未来位置）的注意力分数设为负无穷，
        # 经过softmax后，这些位置的注意力权重会变为0，
        # 从而确保每个位置只能关注自身及之前的位置，不能"偷看"未来的信息。  
        mask_weight = weight.masked_fill(
            self.attention_mask[:seq_size, :seq_size] == 0,
            float('-inf')
        )
        # 4. softmax
        attention_weight = F.softmax(mask_weight, dim=-1)
        # 5. dropout
        attention_weight = self.dropout(attention_weight)
        # 6. 计算注意力值
        attention = attention_weight @ v
        return attention

In [ ]:
summary(SingleHeadAttention(GPTConfig).to('cuda'), (GPTConfig.max_squence_length, GPTConfig.hidden_size))

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super(MultiHeadAttention, self).__init__()
        # 多头注意力
        self.heads = nn.ModuleList([
            SingleHeadAttention(config)
            for _ in range(config.head_num_per_block)
        ])
        # 全连接层
        self.linear = nn.Linear(config.head_num_per_block * config.head_size, config.hidden_size)
        # dropout
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch_size, seq_size, hidden_size)
        # 1. 计算每个头的注意力
        attentions = torch.cat([head(x) for head in self.heads], dim=-1)
        # 2. 通过全连接层
        x = self.linear(attentions)
        # 3. 应用dropout
        x = self.dropout(x)
        return x

In [ ]:
class FeedForward(nn.Module):
    # 在Transformer和大语言模型的上下文中，"Feedforward"或"FFN"（Feed-Forward Network）通常指的是注意力层后的特定结构，这个结构实际上就是一个简单的MLP(多层感知机)。
    def __init__(self, config: GPTConfig):
        super(FeedForward, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size * 4),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_size * 4, config.hidden_size)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch_size, seq_size, hidden_size)
        return self.net(x)

In [ ]:
summary(FeedForward(GPTConfig).to('cuda'), (GPTConfig.max_squence_length, GPTConfig.hidden_size))

In [ ]:
class Block(nn.Module):
    """ Transformer的block（灰色的block） """
    def __init__(self, config: GPTConfig):
        super(Block, self).__init__()
        self.attention    = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)
        # 残差连接
        self.layer_norm1 = nn.LayerNorm(config.hidden_size)
        self.layer_norm2 = nn.LayerNorm(config.hidden_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attention(self.layer_norm1(x))
        x = x + self.feed_forward(self.layer_norm2(x))
        return x


In [ ]:
from typing import Optional, Tuple

In [ ]:
class GPT2(nn.Module):
    """ GPT-2模型 """
    def __init__(self, config: GPTConfig):
        super(GPT2, self).__init__()
        self.config = config
        # 词嵌入
        self.token_embedding_table    = nn.Embedding(config.vocab_size, config.hidden_size)
        # 位置编码
        self.position_embedding_table = nn.Embedding(config.max_squence_length, config.hidden_size)
        # tiktoken处理的是文本到token ID的转换
        # nn.Embedding处理的是token ID到向量表示的转换

        # transformer的block
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.block_size)]
        )

        # 输出层
        self.layer_norm = nn.LayerNorm(config.hidden_size)
        self.linear    = nn.Linear(config.hidden_size, config.vocab_size)
        # 常用的交叉熵损失函数nn.CrossEntropyLoss会内部应用softmax，所以模型本身不需要显式地应用softmax
        # 使用 tie weights 的方法, 词嵌入和输出层共享权重
        self.token_embedding_table.weight = self.linear.weight
        # 权重初始化
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """ 权重初始化 """
        if isinstance(module, nn.Linear):
            # 权重初始化
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            # 偏置初始化
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            # 嵌入层的权重初始化
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, target_ids = None):
        batch, seq_size = input_ids.size()
        """ step1. 编码"""
        # 1. 词嵌入
        token_embedding = self.token_embedding_table(input_ids)
        # 2. 位置编码
        position_embedding = self.position_embedding_table(
            torch.arange(seq_size, device=input_ids.device)
        )
        # 3. 相加
        embedding = token_embedding + position_embedding

        """ step2. transformer的block(灰色) """
        transformer_output = self.blocks(embedding)
        logits = self.linear(self.layer_norm(transformer_output))
        # logits: (batch_size, seq_size, vocab_size)，代表每个token的预测分布

        """ step3. 计算损失 """
        if target_ids is None:
            loss = None
        else:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                target_ids.view(-1)
            )
        return logits, loss


    def generate(self, input_ids: torch.Tensor, max_length: int):
        raise NotImplementedError("生成函数尚未实现")


In [ ]:
import pytorch_lightning as pl

class GPT2Lightning(pl.LightningModule):
    """ GPT-2模型的Lightning封装 """
    def __init__(self, config: GPTConfig):
        super(GPT2Lightning, self).__init__()
        self.model  = GPT2(config)
        self.config = config

    def forward(self, input_ids: torch.Tensor, target_ids: Optional[torch.Tensor] = None):
        return self.model(input_ids, target_ids)

    def training_step(self, batch, batch_idx: int) -> torch.Tensor:
        input_ids, target_ids = batch
        logits, loss = self(input_ids, target_ids)
        # 日志记录
        self.log('train_loss', loss)
        self.log('lr_rate',    self.trainer.optimizers[0].param_groups[0]['lr'])
        return loss

    def validation_step(self, batch, batch_idx: int) -> torch.Tensor:
        input_ids, target_ids = batch
        logits, loss = self(input_ids, target_ids)
        self.log('val_loss', loss)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=3e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)
        return {
            'optimizer'    : optimizer,
            'lr_scheduler' : {
                'scheduler': scheduler,
                'interval' : 'step',  # 每步更新学习率，而不是每个epoch
                'frequency': 1
            }
        }

In [ ]:
monkey_dataset_path = "mobvoi_seq_monkey_general_open_corpus.jsonl"

In [ ]:
!head -n 5 $monkey_dataset_path

In [ ]:
from datasets import load_dataset

In [ ]:
import tiktoken

In [ ]:
import tiktoken
class StreamingGPTDataset(IterableDataset):
    """ 流式数据集 """
    def __init__(self, dataset_path: str, config: GPTConfig, tokenizer="gpt2", split='train', split_ratio=0.9, seed=42):
        self.encoder      = tiktoken.get_encoding(tokenizer)
        self.max_seq_size = config.max_squence_length
        self.eos_token_id = self.encoder.encode(
            "<|endoftext|>",
            allowed_special = {"<|endoftext|>"}
        )[0]
        base_dataset = load_dataset(
            'json',
            data_files = dataset_path,
            streaming  = True,
            split      = "train",
        )
        # 根据分割类型筛选数据
        if split == "train":
            self.dataset = base_dataset.shuffle(seed=seed).filter(
                lambda example, idx: idx % int(1/split_ratio) < 1,
                with_indices = True
            )
        elif split == "val":
            self.dataset = base_dataset.shuffle(seed=seed).filter(
                lambda example, idx: idx % int(1/split_ratio) >= 1,
                with_indices = True
            )
        else:
            self.dataset = base_dataset

    def __iter__(self):
        buffer = []
        for sample in self.dataset:
            # 将一条数据加入buffer
            text         = sample['text']
            encoded_text = self.encoder.encode(text) + [self.eos_token_id]
            buffer.extend(encoded_text)
            # 当缓冲区足够长时，生成训练样本
            while len(buffer) >= self.max_seq_size + 1:
                chunk = buffer[ : self.max_seq_size+1]
                buffer = buffer[self.max_seq_size//2 : ]  # 滑动窗口，重叠50%
                x = torch.tensor(chunk[:-1], dtype=torch.long)
                y = torch.tensor(chunk[1:],  dtype=torch.long)
                yield x, y

In [ ]:
class GPT2DatasetLightning(pl.LightningDataModule):
    """ 数据集 """
    def __init__(self, dataset_path: str, config: GPTConfig, tokenizer="gpt2", split_ratio=0.9, seed=42):
        super(GPT2DatasetLightning, self).__init__()
        self.config       = config
        self.train_dataset  = StreamingGPTDataset(
            dataset_path,
            config,
            tokenizer   = tokenizer,
            split       = 'train',
            split_ratio = split_ratio,
            seed        = seed
        )
        self.val_dataset    = StreamingGPTDataset(
            dataset_path,
            config,
            tokenizer   = tokenizer,
            split       = 'val',
            split_ratio = split_ratio,
            seed        = seed
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size      = self.config.batch_size,
            num_workers     = 1,
            pin_memory      = True,
            prefetch_factor = 2
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size      = self.config.batch_size,
            num_workers     = 1,
            pin_memory      = True,
            prefetch_factor = 2
        )

In [ ]:
data  = GPT2DatasetLightning(monkey_dataset_path, GPTConfig())

In [ ]:
model = GPT2Lightning(GPTConfig())
# 打印模型参数量
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e6:.2f}M")

In [ ]:
# 设置检查点回调
from pytorch_lightning.callbacks import ModelCheckpoint
checkpoint_callback = ModelCheckpoint(
    dirpath    = 'checkpoints',
    filename   = 'model-{epoch:02d}-{val_loss:.4f}',
    save_top_k = 3,  # 保存验证损失最低的3个模型
    monitor    = 'val_loss',
    mode       = 'min',
    save_last  = True  # 也保存最后一个模型
)

In [ ]:
# 创建训练器
from pytorch_lightning.loggers import WandbLogger
import time
logger = WandbLogger(project="GPT2", name=time.strftime("%m%d%H%M"))
trainer = pl.Trainer(
    max_epochs   = 10,  
    accelerator  = 'auto',
    devices      = 1 if torch.cuda.is_available() else None,
    callbacks    = [checkpoint_callback],
    logger       = logger,
    log_every_n_steps = 1,
    num_sanity_val_steps = 0,
)

In [ ]:
trainer.fit(model, data)